# Pretrain on MPOSE2021 -> Scratch/Fine-tune v2

서비스형 hard negative, detector-style augmentation, runtime-alignment metric을 반영한 v2 실험 노트북입니다.


In [ ]:
import ctypes, glob, os
_nv = '/workspace/users/yijin/boot_env/.venv/lib/python3.12/site-packages/nvidia'
if os.path.isdir(_nv):
    for _so in sorted(glob.glob(f'{_nv}/*/lib/*.so.*')):
        try: ctypes.CDLL(_so, mode=ctypes.RTLD_GLOBAL)
        except OSError: pass

import json, sys, time
from argparse import Namespace
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

start = Path.cwd().resolve()
PROJECT_ROOT = None
for base in [start, *start.parents]:
    if (base / 'data' / 'reference_dances').exists() and (base / 'scripts' / 'prepare_mpose2021.py').exists():
        PROJECT_ROOT = base
        break
if PROJECT_ROOT is None:
    raise RuntimeError('project root not found')
sys.path.insert(0, str(PROJECT_ROOT))

import tensorflow as tf
from src.embedding.dataset_contrastive import TARGET_DANCES
from scripts.pretrain_scratch_mpose2021 import train as pretrain_train
from scripts.train_scratch_contrastive import train as scratch_train

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TF version  :', tf.__version__)
print('GPU devices :', tf.config.list_physical_devices('GPU'))


In [ ]:
SEQUENCE_LENGTH = 30
TARGET_DANCES = list(TARGET_DANCES)

SIZES = {
    'small': {
        'embedding_dim': 32,
        'mlp_hidden': 64,
        'tcn_filters': 32, 'tcn_blocks': 3,
        'gcn_filters': 32, 'gcn_blocks': 2,
    },
    'base': {
        'embedding_dim': 64,
        'mlp_hidden': 128,
        'tcn_filters': 64, 'tcn_blocks': 4,
        'gcn_filters': 64, 'gcn_blocks': 3,
    },
}

def _size_kwargs(model, size):
    cfg = dict(SIZES[size])
    if model == 'mlp':
        cfg.setdefault('tcn_filters', 64); cfg.setdefault('tcn_blocks', 4)
        cfg.setdefault('gcn_filters', 64); cfg.setdefault('gcn_blocks', 3)
    elif model == 'tcn':
        cfg.setdefault('mlp_hidden', 128)
        cfg.setdefault('gcn_filters', 64); cfg.setdefault('gcn_blocks', 3)
    else:
        cfg.setdefault('mlp_hidden', 128)
        cfg.setdefault('tcn_filters', 64); cfg.setdefault('tcn_blocks', 4)
    return cfg


## 1. MPOSE2021 Pretrain v2


In [ ]:
PRETRAIN_COMMON = dict(
    data_path=None,
    pose_extractor='movenet',
    split=1,
    output_dir='data/models/pretrain/mpose2021',
    sequence_length=SEQUENCE_LENGTH,
    dropout=0.15,
    epochs=30,
    batch_size=128,
    learning_rate=3e-4,
    min_learning_rate=1e-5,
    warmup_epochs=2,
    patience=10,
    temperature=0.1,
    runtime_jitter=0.01,
    augment_rot_deg=8.0,
    tcn_kernel=3,
    gcn_kernel=9,
    gcn_partition='distance',
    seed=42,
    verbose=2,
    prepare_if_missing=False,
)

PRETRAIN_VARIANTS = [
    {'enabled': True, 'model': m, 'size': s}
    for m in ('mlp', 'tcn', 'gcn')
    for s in ('small', 'base')
]

def run_pretrain_v2(variant):
    cfg = {**PRETRAIN_COMMON, 'model': variant['model']}
    cfg.update(_size_kwargs(variant['model'], variant['size']))
    dim = cfg['embedding_dim']
    cfg['model_name'] = f"mpose_{variant['model']}_{variant['size']}_e{dim}_v2"
    return pretrain_train(Namespace(**cfg))

pretrain_paths = {}
pretrain_results = {}
for i, v in enumerate([x for x in PRETRAIN_VARIANTS if x.get('enabled', True)]):
    key = (v['model'], v['size'])
    print(f"\n[PRETRAIN {i+1}/{len(PRETRAIN_VARIANTS)}] {key}")
    res = run_pretrain_v2(v)
    pretrain_results[key] = res
    pretrain_paths[key] = res['paths']['weights']


## 2. Scratch Contrastive v2


In [ ]:
SCRATCH_COMMON_V2 = dict(
    data_dir='data/reference_dances',
    dances=TARGET_DANCES,
    output_dir='data/models/scratch',
    feature_dims=2,
    sequence_length=SEQUENCE_LENGTH,
    val_fraction=0.15,
    dropout=0.15,
    epochs=50,
    steps_per_epoch=200,
    validation_steps=30,
    batch_size=128,
    learning_rate=1e-3,
    min_learning_rate=1e-5,
    warmup_epochs=3,
    patience=10,
    temperature=0.1,
    triplet_margin=0.2,
    positive_jitter=2,
    negative_gap=90,
    false_negative_gap=4,
    hard_negative_min_gap=6,
    hard_negative_max_gap=24,
    hard_negative_prob=0.5,
    cross_song_prob=0.5,
    runtime_jitter=0.005,
    joint_dropout_prob=0.04,
    frame_hold_prob=0.05,
    temporal_warp_prob=0.25,
    temporal_warp_strength=0.15,
    eval_max_samples=128,
    eval_tolerance_frames=12,
    eval_candidate_stride=3,
    eval_user_runtime_jitter=0.01,
    eval_user_joint_dropout_prob=0.04,
    eval_user_frame_hold_prob=0.05,
    eval_user_temporal_warp_prob=0.25,
    eval_user_temporal_warp_strength=0.15,
    pretrained_weights=None,
    name_suffix='',
    seed=42,
    no_quantize=False,
    keep_checkpoint=False,
    verbose=2,
)

SCRATCH_VARIANTS_V2 = [
    {'enabled': True, 'model': m, 'size': s, 'loss': l}
    for m in ('mlp', 'tcn', 'gcn')
    for s in ('small', 'base')
    for l in ('infonce', 'triplet')
]

def run_scratch_v2(variant):
    cfg = {**SCRATCH_COMMON_V2, 'model': variant['model'], 'loss': variant['loss']}
    cfg.update(_size_kwargs(variant['model'], variant['size']))
    dim = cfg['embedding_dim']
    cfg['model_name'] = f"scratch_{variant['model']}_{variant['size']}_{variant['loss']}_e{dim}_v2"
    return scratch_train(Namespace(**cfg))

scratch_results_v2 = {}
for i, v in enumerate([x for x in SCRATCH_VARIANTS_V2 if x.get('enabled', True)]):
    key = (v['model'], v['size'], v['loss'])
    print(f"\n[SCRATCH V2 {i+1}/{len(SCRATCH_VARIANTS_V2)}] {key}")
    scratch_results_v2[key] = run_scratch_v2(v)


## 3. Fine-tune v2 (same trainer, pretrained weights loaded)


In [ ]:
FINETUNE_COMMON_V2 = dict(
    data_dir='data/reference_dances',
    dances=TARGET_DANCES,
    output_dir='data/models/embedding',
    feature_dims=2,
    sequence_length=SEQUENCE_LENGTH,
    val_fraction=0.15,
    dropout=0.15,
    epochs=40,
    steps_per_epoch=200,
    validation_steps=30,
    batch_size=128,
    learning_rate=1e-4,
    min_learning_rate=1e-6,
    warmup_epochs=1,
    patience=8,
    temperature=0.1,
    triplet_margin=0.2,
    positive_jitter=2,
    negative_gap=90,
    false_negative_gap=4,
    hard_negative_min_gap=6,
    hard_negative_max_gap=24,
    hard_negative_prob=0.5,
    cross_song_prob=0.5,
    runtime_jitter=0.005,
    joint_dropout_prob=0.04,
    frame_hold_prob=0.05,
    temporal_warp_prob=0.25,
    temporal_warp_strength=0.15,
    eval_max_samples=128,
    eval_tolerance_frames=12,
    eval_candidate_stride=3,
    eval_user_runtime_jitter=0.01,
    eval_user_joint_dropout_prob=0.04,
    eval_user_frame_hold_prob=0.05,
    eval_user_temporal_warp_prob=0.25,
    eval_user_temporal_warp_strength=0.15,
    pretrained_weights=None,
    name_suffix='',
    seed=42,
    no_quantize=False,
    keep_checkpoint=False,
    verbose=2,
)

FINETUNE_VARIANTS_V2 = SCRATCH_VARIANTS_V2

def run_finetune_v2(variant, pretrained_path):
    cfg = {**FINETUNE_COMMON_V2, 'model': variant['model'], 'loss': variant['loss']}
    cfg.update(_size_kwargs(variant['model'], variant['size']))
    dim = cfg['embedding_dim']
    cfg['model_name'] = f"finetune_{variant['model']}_{variant['size']}_{variant['loss']}_e{dim}_v2"
    cfg['pretrained_weights'] = pretrained_path
    return scratch_train(Namespace(**cfg))

finetune_results_v2 = {}
for i, v in enumerate([x for x in FINETUNE_VARIANTS_V2 if x.get('enabled', True)]):
    key = (v['model'], v['size'], v['loss'])
    pretrained_key = (v['model'], v['size'])
    if pretrained_key not in pretrain_paths:
        print(f"[SKIP] {key}: missing pretrain weights")
        continue
    print(f"\n[FINETUNE V2 {i+1}/{len(FINETUNE_VARIANTS_V2)}] {key}")
    finetune_results_v2[key] = run_finetune_v2(v, pretrain_paths[pretrained_key])


## 4. Compare v2 metrics


In [ ]:
def _final(history, keys):
    for k in keys:
        if k in history and history[k]:
            return history[k][-1]
    return float('nan')

rows = []
for v in FINETUNE_VARIANTS_V2:
    key = (v['model'], v['size'], v['loss'])
    for regime, res in [('scratch_v2', scratch_results_v2.get(key)), ('finetune_v2', finetune_results_v2.get(key))]:
        if res is None:
            continue
        hist = res.get('history', {})
        summary = res.get('training_summary', {})
        smoke = res.get('smoke_metrics', {})
        align = res.get('runtime_alignment_metrics', {})
        rows.append({
            'model': v['model'],
            'size': v['size'],
            'loss': v['loss'],
            'regime': regime,
            'epochs_run': len(hist.get('loss', [])),
            'best_val_loss': summary.get('best_val_loss'),
            'best_epoch': summary.get('best_epoch'),
            'within_margin': smoke.get('within_dance_margin_mean'),
            'cross_margin': smoke.get('cross_dance_margin_mean'),
            'runtime_top1': align.get('top1_acc'),
            'runtime_top3': align.get('top3_acc'),
            'runtime_margin': align.get('target_margin_mean'),
            'runtime_rank_mean': align.get('target_rank_mean'),
        })
summary_v2 = pd.DataFrame(rows).sort_values(['model', 'size', 'loss', 'regime'])
summary_v2


In [ ]:
fig, axes = plt.subplots(len(FINETUNE_VARIANTS_V2), 2, figsize=(14, 2.2 * len(FINETUNE_VARIANTS_V2)))
if len(FINETUNE_VARIANTS_V2) == 1:
    axes = axes[None, :]

for row, v in enumerate(FINETUNE_VARIANTS_V2):
    key = (v['model'], v['size'], v['loss'])
    rs = scratch_results_v2.get(key)
    rf = finetune_results_v2.get(key)
    title = f"{v['model']}/{v['size']}/{v['loss']}"

    ax = axes[row, 0]
    if rs is not None:
        h = rs['history']
        ax.plot(h.get('loss', []), 'b-', label='scratch train', alpha=0.85)
        if 'val_loss' in h:
            ax.plot(h['val_loss'], 'b--', alpha=0.55, label='scratch val')
    if rf is not None:
        h = rf['history']
        ax.plot(h.get('loss', []), 'r-', label='finetune train', alpha=0.85)
        if 'val_loss' in h:
            ax.plot(h['val_loss'], 'r--', alpha=0.55, label='finetune val')
    ax.set_title(f'{title} - loss')
    ax.set_xlabel('epoch')
    ax.legend(fontsize=7, loc='upper right')

    ax = axes[row, 1]
    metric = 'retrieval_acc' if v['loss'] == 'infonce' else 'pos_margin'
    val_metric = f'val_{metric}'
    if rs is not None:
        h = rs['history']
        if metric in h:
            ax.plot(h[metric], 'b-', label='scratch', alpha=0.85)
        if val_metric in h:
            ax.plot(h[val_metric], 'b--', alpha=0.55)
    if rf is not None:
        h = rf['history']
        if metric in h:
            ax.plot(h[metric], 'r-', label='finetune', alpha=0.85)
        if val_metric in h:
            ax.plot(h[val_metric], 'r--', alpha=0.55)
    ax.set_title(f'{title} - {metric}')
    ax.set_xlabel('epoch')
    ax.legend(fontsize=7, loc='lower right')

plt.tight_layout()
plt.show()


## 5. Optional: exported TFLite runtime-alignment check


In [ ]:
from scripts.eval_runtime_alignment import evaluate_runtime_alignment

example_variant = {'model': 'tcn', 'size': 'base', 'loss': 'infonce'}
example_name = f"finetune_{example_variant['model']}_{example_variant['size']}_{example_variant['loss']}_e{SIZES[example_variant['size']]['embedding_dim']}_v2"
args = Namespace(
    kind='embedding',
    model_name=example_name,
    model_dir='data/models/embedding',
    model_path=None,
    data_dir='data/reference_dances',
    dances=TARGET_DANCES,
    feature_dims=2,
    sequence_length=SEQUENCE_LENGTH,
    candidate_stride=3,
    tolerance_frames=12,
    samples=128,
    seed=42,
    user_runtime_jitter=0.01,
    user_joint_dropout_prob=0.04,
    user_frame_hold_prob=0.05,
    user_temporal_warp_prob=0.25,
    user_temporal_warp_strength=0.15,
    output=None,
)
evaluate_runtime_alignment(args)
